# YouTube Comment Scraper — yt-dlp Edition
### HPDP Project 2 — Real-Time Sentiment Analysis
**Target Video:** https://youtu.be/dV-H1EKmCxA

**Output:** `youtube_comments_raw.csv` — raw, untouched comments.

---
> ✅ No API key required. Uses `yt-dlp` which handles YouTube's internal API automatically.
> yt-dlp is actively maintained and adapts to YouTube changes — much more reliable than manually parsing `ytInitialData`.


## Step 1 — Configuration

In [ ]:
import os

# Video Target
VIDEO_ID  = 'dV-H1EKmCxA'
VIDEO_URL = f'https://www.youtube.com/watch?v={VIDEO_ID}'

# Scrape Settings
MAX_COMMENTS = None

# Output
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = f'{OUTPUT_DIR}/youtube_comments_raw.csv'

print(f'🎯 Target  : {VIDEO_URL}')
print(f'💬 Max     : {MAX_COMMENTS or "ALL"} comments')
print(f'💾 Output  : {OUTPUT_CSV}')


🎯 Target  : https://www.youtube.com/watch?v=dV-H1EKmCxA
💬 Max     : ALL comments
💾 Output  : /content/output/youtube_comments_raw.csv


## Step 2 — Install & Import Libraries

In [ ]:
!pip install yt-dlp --quiet
print('✅ yt-dlp ready!')


✅ yt-dlp ready!


In [ ]:
import os
import yt_dlp
import pandas as pd
from datetime import datetime, timezone

print('✅ All imports successful!')


✅ All imports successful!


## Step 3 — Fetch Video Metadata + Comments

`yt-dlp` fetches both metadata and comments in one call using `getcomments=True`.
It handles pagination, continuation tokens, and YouTube API changes internally.


In [ ]:
def scrape_youtube(video_url, max_comments=None):
    ydl_opts = {
        'quiet'       : True,
        'no_warnings' : True,
        'skip_download': True,
        'getcomments' : True,
        'extractor_args': {
            'youtube': {
                'max_comments': [f'{max_comments if max_comments else 99999},100,0,0'],
            }
        },
    }

    print(f'🚀 Fetching video info and comments...')
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

    return info


info = scrape_youtube(VIDEO_URL, max_comments=MAX_COMMENTS)

# ── Metadata ──────────────────────────────────────────────────────────────────
metadata = {
    'video_id'     : info.get('id'),
    'video_url'    : f"https://youtu.be/{info.get('id')}",
    'title'        : info.get('title'),
    'channel'      : info.get('uploader'),
    'view_count'   : info.get('view_count'),
    'like_count'   : info.get('like_count'),
    'comment_count': info.get('comment_count'),
    'scraped_at'   : datetime.now(timezone.utc).isoformat(),
}

print('\n📹 VIDEO METADATA')
print('─' * 45)
for k, v in metadata.items():
    if v is not None:
        print(f'  {k:<15}: {v:,}' if isinstance(v, int) else f'  {k:<15}: {v}')
print('─' * 45)


🚀 Fetching video info and comments...

📹 VIDEO METADATA
─────────────────────────────────────────────
  video_id       : dV-H1EKmCxA
  video_url      : https://youtu.be/dV-H1EKmCxA
  title          : Geography Now! MALAYSIA
  channel        : Geography Now
  view_count     : 3,587,590
  like_count     : 100,810
  comment_count  : 18,805
  scraped_at     : 2026-06-06T13:36:46.897227+00:00
─────────────────────────────────────────────


## Step 4 — Build DataFrame

In [ ]:
raw_comments = info.get('comments') or []
print(f'📦 Raw comments fetched: {len(raw_comments):,}')

rows = []
for c in raw_comments:
    rows.append({
        'comment_id'   : c.get('id', ''),
        'parent_id'    : None if c.get('parent') == 'root' else c.get('parent'),
        'comment_type' : 'top_level' if c.get('parent') == 'root' else 'reply',
        'author'       : c.get('author', ''),
        'text'         : (c.get('text') or '').strip(),
        'like_count'   : c.get('like_count', 0),
        'reply_count'  : 0,
        'published_at' : datetime.fromtimestamp(c['timestamp'], tz=timezone.utc).isoformat()
                         if c.get('timestamp') else '',
        'video_id'     : metadata['video_id'],
    })

df_raw = pd.DataFrame(rows)

reply_counts = df_raw[df_raw['comment_type'] == 'reply'].groupby('parent_id').size()
df_raw['reply_count'] = df_raw['comment_id'].map(reply_counts).fillna(0).astype(int)

print(f'DataFrame shape: {df_raw.shape}')
df_raw.head()


📦 Raw comments fetched: 18,805
DataFrame shape: (18805, 9)


,comment_id,parent_id,comment_type,author,text,like_count,reply_count,published_at,video_id
0,UgylxmmbT-YO8_OFn6l4AaABAg,None,top_level,@GeographyNow,What's that? Your country only has ONE King?.....,6100,310,2018-06-07T00:00:00+00:00,dV-H1EKmCxA
1,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,UgylxmmbT-YO8_OFn6l4AaABAg,reply,@Bluebeanzz,Technically is 8 sultan and 1 raja (Perlis)…,53,2,2018-06-07T00:00:00+00:00,dV-H1EKmCxA
2,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D97vuIuj3y_1,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,reply,@Darklord-ms1if,@Bluebeanzz well that's my village,0,0,2020-06-07T00:00:00+00:00,dV-H1EKmCxA
3,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D9D88cnHcori,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h97X3H60Qa,reply,@hideriplays2626,"@Bluebeanzz 7 sultans, 1 raja (Perlis), and 1...",0,0,2021-06-07T00:00:00+00:00,dV-H1EKmCxA
4,UgylxmmbT-YO8_OFn6l4AaABAg.8h94Igzpd4D8h94bhH_dvC,UgylxmmbT-YO8_OFn6l4AaABAg,reply,@allysuke,Why didn’t u talk about the F1 circuit,144,2,2018-06-07T00:00:00+00:00,dV-H1EKmCxA


## Step 5 — Save Raw CSV & Download
> No cleaning done here. The `text` column is exactly what was scraped from YouTube.


In [ ]:
df_raw.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ Raw CSV saved → {OUTPUT_CSV}')
print(f'   Rows    : {len(df_raw):,}')
print(f'   Columns : {list(df_raw.columns)}')
print(f'   Size    : {os.path.getsize(OUTPUT_CSV)/1024:.1f} KB')


✅ Raw CSV saved → /content/output/youtube_comments_raw.csv
   Rows    : 18,805
   Columns : ['comment_id', 'parent_id', 'comment_type', 'author', 'text', 'like_count', 'reply_count', 'published_at', 'video_id']
   Size    : 3776.8 KB


In [ ]:
from google.colab import files

print('⬇️  Downloading raw CSV...')
files.download(OUTPUT_CSV)
print('✅ Done! Check your Downloads folder.')
print('\n➡️  Next: Upload this file to Notebook 2 for preprocessing.')


⬇️  Downloading raw CSV...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Check your Downloads folder.

➡️  Next: Upload this file to Notebook 2 for preprocessing.
